# Day 11 — 실험 설계: 반복, 배치, 교란, 대조군

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-2/day-11.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-2/day-11.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 가상의 발현 표에서 batch effect 찾기

샘플 12개(GC 6, FLT 6)를 두 배치로 나눠 처리한 가상 데이터를 만듭니다. 유전자 200개 중 100개는 배치 B2에서 1.0만큼 높고, 20개는 FLT에서 0.8만큼 높습니다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

rng = np.random.default_rng(11)
n_genes = 200

meta = pd.DataFrame({
    "sample": [f"S{i:02d}" for i in range(1, 13)],
    "condition": ["GC"] * 6 + ["FLT"] * 6,
    "batch": ["B1", "B2"] * 6,            # 실험 날짜가 두 번으로 나뉘었다고 가정
}).set_index("sample")

gene_mean = rng.normal(8, 1, n_genes)                         # 유전자마다 다른 기본 발현량
expr = gene_mean + rng.normal(0, 0.3, size=(12, n_genes))     # 샘플 12개 × 유전자 200개
expr[(meta["batch"] == "B2").to_numpy(), :100] += 1.0         # 배치 효과: 유전자 100개가 B2에서 +1.0
expr[(meta["condition"] == "FLT").to_numpy(), :20] += 0.8     # 생물학적 효과: 유전자 20개가 FLT에서 +0.8
expr = pd.DataFrame(expr, index=meta.index, columns=[f"gene{i:03d}" for i in range(n_genes)])

pca = PCA(n_components=2)
pcs = pca.fit_transform(expr - expr.mean())                   # 유전자별 평균을 빼고 PCA
meta["PC1"], meta["PC2"] = pcs[:, 0], pcs[:, 1]

print("설명 분산 비율:", pca.explained_variance_ratio_.round(2))
print(meta.groupby("batch")["PC1"].mean().round(2))
print(meta.groupby("condition")["PC1"].mean().round(2))

### 예시 2 — 설계식에 batch를 넣으면 무엇이 달라지나

예시 1을 실행한 같은 노트북에서 이어서 실행합니다(`meta`, `expr`를 그대로 씁니다).

In [ ]:
import statsmodels.formula.api as smf

d = meta.assign(y=expr["gene005"])        # gene005: 배치 효과와 조건 효과를 둘 다 받는 유전자

m1 = smf.ols("y ~ condition", data=d).fit()
m2 = smf.ols("y ~ batch + condition", data=d).fit()

term = "condition[T.GC]"                  # FLT가 기준(알파벳 순), GC가 FLT보다 얼마나 다른가
for name, m in [("y ~ condition        ", m1), ("y ~ batch + condition", m2)]:
    print(f"{name}: 계수={m.params[term]:.2f}, 표준오차={m.bse[term]:.2f}, p={m.pvalues[term]:.4f}")

### 예시 3 — 교차표로 교란을 미리 잡기

In [ ]:
import pandas as pd

balanced = pd.DataFrame({"condition": ["GC"] * 6 + ["FLT"] * 6, "batch": ["B1", "B2"] * 6})
confounded = pd.DataFrame({"condition": ["GC"] * 6 + ["FLT"] * 6, "batch": ["B1"] * 6 + ["B2"] * 6})

print("균형 잡힌 설계")
print(pd.crosstab(balanced["condition"], balanced["batch"]))
print("\n교란된 설계")
print(pd.crosstab(confounded["condition"], confounded["batch"]))

## 직접 해보기

1. `notebooks/02_statistics_basics.ipynb` 맨 아래에 `## Day 11 — 실험 설계와 batch effect` 섹션을 만들고 예시 1을 실행합니다.
2. 먼저 메타데이터부터 봅니다. `pd.crosstab(meta["condition"], meta["batch"])`를 실행해 설계가 균형인지 확인합니다(네 칸 모두 3이어야 합니다).
3. 같은 PCA 결과를 색만 바꿔 두 번 그립니다.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.scatterplot(data=meta, x="PC1", y="PC2", hue="batch", s=80, ax=axes[0])
sns.scatterplot(data=meta, x="PC1", y="PC2", hue="condition", s=80, ax=axes[1])
plt.tight_layout()
plt.savefig("../figures/day11_batch_pca.png", dpi=150)
plt.show()